# TopoLS Tutorial

### Step 1: Compile a quantum circuit using TopoLS

In [1]:
# Let's compile 16 qubits GHZ circuit
!python3 prog.py -f ghz_16 -b 20 -zx 1 -dir 1 -l 4 -r 0 -s 2 -t 2 -i 1000 -csv result -sp 0 -b0 0
# Compilation results are saved in result/topols/

Executing ghz_16 benchmark.
Embedding progress:
 67%|██████████████████████████████               | 2/3 [00:12<00:06,  6.40s/it]
x_length: 9.0 y_length: 9.0 z_length: 3
Space-time volume: 243.0
Time: 3
Space: 81.0
Compilation time: 12.804823398590088


#### We explain the arguments here.

-b: Maximum block size for circuit slicing. Although a topology-aware partitioning is used, this argument specifies the upper limit on the size of each block.

-zx: ZX optimization level (0: off, 1: on). Enabling this option activates spider fusion.

-dir: Direction optimization level (0: off, 1: on). Enable this option activates placement exploration in MCTS embedding.

-l: Qubit number per row. Logical qubits are initially placed in a rectangular layout, and this value specifies how many logical qubits are placed in each row.

-r: Initial random seed for circuit compilation.

-s: Number of random seed will be tried.

-t: Time bound for each MCTS iteration (second).

-i: Number of iterations for MCTS.

-csv: csv result file name, saved in result/topols/.

#### If the circuit is too dense:

-sp: For dense circuit we will spread the quantum gates into different rows. This parameter specifies how many gates are placed in each row. In such cases, it is also recommended to reduce -b to 1 or 2.

-b0: Inforce the first row of circuit as a block (0: off, 1: on).

In [2]:
# Dense circuit example
!python3 prog.py -f random_500 -b 1 -zx 1 -dir 1 -l 23 -r 0 -s 2 -t 5 -i 1000 -csv result -sp 70 -b0 0
# The baseline method (DASCOT) achieves a z-length of 43, whereas our method reduces it to 19 with the same spatial footprint, representing a significant improvement.

Executing random_500 benchmark.
Embedding progress:
 89%|████████████████████████████████████████     | 8/9 [03:34<00:26, 26.85s/it]
x_length: 47.0 y_length: 45.0 z_length: 19
Space-time volume: 40185.0
Time: 19
Space: 2115.0
Compilation time: 214.77749347686768


### Step 2: Transform to tqec bgraph format, and visualization

In [3]:
# Using 2tqec to transpile the TopoLS compilation result to a TQEC readable format, and visualize the TQEC schedule
# -f specifies the filename of the .pkl file to be used for translation, -p specifies whether to visualize the TQEC schedule. The .png file of the pipe diagram is saved in result/visualizatioin/
!python3 2tqec.py -f ghz_16 -p True
!python3 2tqec.py -f random_500 -p True

100%|████████████████████████████████████████| 137/137 [00:00<00:00, 250.70it/s]
Figure(1800x1800)
100%|█████████████████████████████████████| 12005/12005 [22:23<00:00,  8.93it/s]
Figure(1800x1800)


### Step 3: Simulate our pipe using TQEC
Note: TQEC is still developing methods for simulating S, T, and (in some cases) H gates. 

Therefore, we recommend simulating only circuits composed of CNOT gates.

In [4]:
# Given the runtime constraints, we only simulate a CNOT here.
!python3 prog.py -f CNOT -b 20 -zx 1 -dir 1 -l 4 -r 0 -s 2 -t 2 -i 1000 -csv result -sp 0 -b0 0
!python3 2tqec.py -f CNOT -p True

# -f specifies the filename of the .bgraph file to be simulated, which is generated by 2tqec.py. The .bgraph file should be placed in result/bgraph/ directory.
# Simulation results are saved in result/simulation/
!python3 pipe_sim.py -f CNOT

Executing CNOT benchmark.
Embedding progress:
 50%|██████████████████████▌                      | 1/2 [00:00<00:00,  3.96it/s]
x_length: 5.0 y_length: 3.0 z_length: 2
Space-time volume: 30.0
Time: 2
Space: 15.0
Compilation time: 0.25355100631713867
100%|██████████████████████████████████████████| 11/11 [00:00<00:00, 408.35it/s]
Figure(1800x1800)
Computing minimum fill of ports to reduce simulation times.
Performing simulations...
Starting 16 workers...
30 tasks left:
  workers    decoder eta shots_left errors_left json_metadata                   
        1 pymatching <1m     991535        9999 d=3,r=3,p=0.0001                
        1 pymatching   ?    1000000       10000 d=3,r=3,p=0.00021544346900318845
        1 pymatching   ?    1000000       10000 d=3,r=3,p=0.00046415888336127773
        1 pymatching   ?    1000000       10000 d=3,r=3,p=0.001                 
        1 pymatching   ?    1000000       10000 d=3,r=3,p=0.002154434690031882  
        1 pymatching   ?    1000000       